# RCAEval RE2-OB: загрузка данных

In [1]:
import json
import random
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 12)

## 1. Путь к данным

Папка `data/` исключена из Git. Ноутбук можно запускать из корня репозитория или из `notebooks/`.

In [8]:
PROJECT_ROOT = Path('./../')
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "rcaeval"
REVISION = "afeacb11bcc94dadfd1c8f483ee4377b2b8b614e"
manifest_path = DATA_DIR / "download_manifest.json"

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest

{'repo_id': 'phamquiluan/RCAEval',
 'revision': 'afeacb11bcc94dadfd1c8f483ee4377b2b8b614e',
 'subset': 'RE2-OB',
 'case_count': 90,
 'total_bytes': 922514305,
 'files': [{'path': 'cases.parquet',
   'bytes': 29500,
   'sha256': 'c49a288920dbba2e8e724679a14636d5c7eb2b45426bba14007ef79a6c0ab1bb'},
  {'path': 're2ob_checkoutservice_cpu_1/inject_time.txt',
   'bytes': 10,
   'sha256': 'c17fcaa1047c07738176849000c1b88e1f0ddda57a2a73a42ab0c8c58c83e0a3'},
  {'path': 're2ob_checkoutservice_cpu_1/logs.parquet',
   'bytes': 336876,
   'sha256': 'e1a25e50c8b0beae4b2886df54d86d2f28877d0b611fed98411e9c0b0c77dad3'},
  {'path': 're2ob_checkoutservice_cpu_1/metrics.parquet',
   'bytes': 157212,
   'sha256': 'f46b3d354b234e37c424f54a005329b61aac83d15dcdd46b5642e55f390ab25a'},
  {'path': 're2ob_checkoutservice_cpu_1/traces.parquet',
   'bytes': 10146857,
   'sha256': '56ca8f6dbeb76fab2d33faeca54bce8f40e9c5491ebd76404ebb2dfd83409367'},
  {'path': 're2ob_checkoutservice_cpu_2/inject_time.txt',
   'bytes':

## 2. Индекс инцидентов

`cases.parquet` содержит индекс всего RCAEval. Оставляем только `RE2-OB`; телеметрию остальных подмножеств не скачиваем.

In [9]:
case_index = pd.read_parquet(DATA_DIR / "cases.parquet")
cases = (
    case_index.loc[case_index["dataset"].eq("RE2-OB")]
    .sort_values("case")
    .reset_index(drop=True)
)

required_files = (
    "metrics.parquet",
    "logs.parquet",
    "traces.parquet",
    "inject_time.txt",
)

## 3. Смотрим данные

In [15]:
cases['root_cause_service'].unique()

<ArrowStringArray>
[      'checkoutservice',       'currencyservice',          'emailservice',
 'productcatalogservice', 'recommendationservice']
Length: 5, dtype: str

In [16]:
cases['fault'].unique()

<ArrowStringArray>
['cpu', 'delay', 'disk', 'loss', 'mem', 'socket']
Length: 6, dtype: str

# Сплитим на dev val test

Разбиваем сплиты по service X fault, чтобы выяснить обощающую способность агента.

In [21]:
services = sorted(cases["root_cause_service"].unique())
faults = sorted(cases["fault"].unique())

rng = random.Random(42)
rng.shuffle(services)
rng.shuffle(faults)

split_by_pair = {}
for i, fault in enumerate(faults):
    for j, service in enumerate(services):
        split = "development"
        if j == i % len(services):
            split = "validation"
        elif j == (i + 1) % len(services):
            split = "test"
        split_by_pair[(service, fault)] = split

cases["split"] = [
    split_by_pair[(service, fault)]
    for service, fault in zip(cases["root_cause_service"], cases["fault"], strict=True)
]

In [25]:
cases.groupby("split").agg(count=("case", 'count'))

,count
split,
development,54
test,18
validation,18


## 4. Чтение одного инцидента

Каждый случай содержит метрики, логи, трассы и время внедрения сбоя.

In [28]:
development_cases = cases.loc[cases["split"].eq("development")].copy()
CASE_ID = development_cases.iloc[0]["case"]

case_dir = DATA_DIR / CASE_ID

metrics = pd.read_parquet(case_dir / "metrics.parquet")
logs = pd.read_parquet(case_dir / "logs.parquet")
traces = pd.read_parquet(case_dir / "traces.parquet")
inject_time = int((case_dir / "inject_time.txt").read_text(encoding="utf-8").strip())
case_metadata = development_cases.loc[development_cases["case"].eq(CASE_ID)]


# метрики

In [43]:
display(
    metrics.head(), 
    metrics.describe(), 
    metrics.dtypes
)

,time,adservice_cpu,cartservice_cpu,checkoutservice_cpu,currencyservice_cpu,emailservice_cpu,...,emailservice_latency-90,frontend_latency-90,paymentservice_latency-90,productcatalogservice_latency-90,recommendationservice_latency-90,shippingservice_latency-90
0,1705353846,0.685023,2.216467,0.215886,19.035876,0.279166,...,0.0046,0.819737,0.008955,0.004418,0.009670,0.004576
1,1705353847,0.685023,2.216467,0.380058,19.035876,0.279166,...,0.0046,0.823697,0.008955,0.004423,0.009672,0.004576
2,1705353848,0.582707,2.216467,0.380058,19.035876,0.279166,...,0.0046,0.823697,0.008636,0.004423,0.009672,0.004576
3,1705353849,0.625903,2.216467,0.380058,19.106596,0.279166,...,0.0046,0.814381,0.008636,0.004416,0.009672,0.004576
4,1705353850,0.613414,2.216467,0.380058,19.106596,0.279166,...,0.0046,0.814381,0.008636,0.004416,0.009672,0.004576


,time,adservice_cpu,cartservice_cpu,checkoutservice_cpu,currencyservice_cpu,emailservice_cpu,...,emailservice_latency-90,frontend_latency-90,paymentservice_latency-90,productcatalogservice_latency-90,recommendationservice_latency-90,shippingservice_latency-90
count,1.441000e+03,1441.000000,1441.000000,1441.000000,1441.000000,1441.000000,...,1441.000000,1441.000000,1441.000000,1441.000000,1441.000000,1441.000000
mean,1.705355e+09,0.603192,2.133318,9.365744,18.149433,0.288312,...,0.004643,0.651630,0.007881,0.004428,0.009650,0.004589
std,4.161252e+02,0.080557,0.253714,9.330049,1.672591,0.031014,...,0.000067,0.075788,0.000917,0.000025,0.000079,0.000024
min,1.705354e+09,0.250993,1.257458,0.185900,9.656963,0.165910,...,0.004600,0.486947,0.004938,0.004378,0.009522,0.004553
25%,1.705354e+09,0.556546,1.974268,0.417569,18.385227,0.278798,...,0.004600,0.594118,0.007500,0.004411,0.009585,0.004571
50%,1.705355e+09,0.624005,2.145418,0.641607,18.741145,0.292722,...,0.004600,0.665625,0.008042,0.004425,0.009646,0.004583
75%,1.705355e+09,0.657439,2.321992,19.992560,19.017717,0.306109,...,0.004686,0.707661,0.008550,0.004442,0.009696,0.004601
max,1.705355e+09,0.765039,2.734659,20.031263,19.597709,0.345321,...,0.004862,0.830861,0.009643,0.004514,0.009912,0.004676


time                                  int64
adservice_cpu                       float64
cartservice_cpu                     float64
checkoutservice_cpu                 float64
currencyservice_cpu                 float64
                                     ...   
frontend_latency-90                 float64
paymentservice_latency-90           float64
productcatalogservice_latency-90    float64
recommendationservice_latency-90    float64
shippingservice_latency-90          float64
Length: 73, dtype: object

## проверим пропуски

In [60]:
metrics_with_gaps = metrics[metrics.isna().any(axis=1)]

display(metrics_with_gaps)
display(metrics_with_gaps.columns[metrics_with_gaps.isna().any()].tolist())

,time,adservice_cpu,cartservice_cpu,checkoutservice_cpu,currencyservice_cpu,emailservice_cpu,...,emailservice_latency-90,frontend_latency-90,paymentservice_latency-90,productcatalogservice_latency-90,recommendationservice_latency-90,shippingservice_latency-90
179,1705354025,0.632431,1.834203,0.350268,14.372005,0.29030,...,0.00460,0.708671,0.007667,0.004415,0.009670,0.004584
643,1705354489,0.654832,1.471211,0.464196,19.074129,0.16591,...,0.00468,0.730224,0.005600,0.004402,0.009617,0.004570


['recommendationservice_cpu',
 'redis_cpu',
 'recommendationservice_diskio',
 'redis_diskio']

# логи

In [ ]:
display(
    logs.head(), 
    logs.describe(), 
    logs.dtypes
)

,timestamp,container_name,message
0,1705353846,currencyservice,conversion request successful
1,1705353846,currencyservice,conversion request successful
2,1705353846,frontend,request started
3,1705353846,frontend,serving product page
4,1705353846,currencyservice,conversion request successful


,timestamp
count,171322.0
mean,1705354564.669307
std,413.548652
min,1705353846.0
25%,1705354209.0
50%,1705354565.0
75%,1705354920.0
max,1705355286.0


timestamp          Int64
container_name    string
message           string
dtype: object

# проверим пропуски

In [61]:
logs_with_gaps = logs[logs.isna().any(axis=1)]

display(logs_with_gaps)
display(logs_with_gaps.columns[logs_with_gaps.isna().any()].tolist())

,timestamp,container_name,message


[]

# Трассы

In [64]:
display(
    traces.head(), 
    traces.describe(), 
    traces.dtypes
)

,time,traceID,spanID,serviceName,methodName,operationName,parentSpanID,startTimeMillis,startTime,duration,statusCode
0,21:24,46d5d1e1fc25a09db4a81d4d105cee3d,de79080386c4c5c3,currencyservice,Convert,grpc.hipstershop.CurrencyService/Convert,433c41ba31b519dd,1705353846065,1705353846065999,186,0
1,21:24,46d5d1e1fc25a09db4a81d4d105cee3d,433c41ba31b519dd,frontendservice,Convert,hipstershop.CurrencyService/Convert,6de46372468bc931,1705353846065,1705353846065431,4133,0
2,21:24,46d5d1e1fc25a09db4a81d4d105cee3d,28de2c1a557175ec,frontendservice,Convert,hipstershop.CurrencyService/Convert,6de46372468bc931,1705353846069,1705353846069657,3587,0
3,21:24,46d5d1e1fc25a09db4a81d4d105cee3d,a6ccc9fc0e441e5c,currencyservice,Convert,grpc.hipstershop.CurrencyService/Convert,28de2c1a557175ec,1705353846071,1705353846071000,134,0
4,21:24,26e74eeaa91cde16a44883aa36a33007,926d999ebf94ce86,frontendservice,<NA>,frontend,<NA>,1705353846073,1705353846073074,222894,<NA>


,startTimeMillis,startTime,duration,statusCode
count,391997.0,391997.0,391997.0,373623.0
mean,1705354564208.377686,1705354564208854.25,23756.333765,0.0
std,414441.259253,414441260.189825,83163.888682,0.0
min,1705353846065.0,1705353846065431.0,2.0,0.0
25%,1705354207282.0,1705354207282584.0,39.0,0.0
50%,1705354563863.0,1705354563863000.0,2301.0,0.0
75%,1705354921185.0,1705354921185992.0,4312.0,0.0
max,1705355285970.0,1705355285970988.0,2396393.0,0.0


time               string
traceID            string
spanID             string
serviceName        string
methodName         string
operationName      string
parentSpanID       string
startTimeMillis     Int64
startTime           Int64
duration            Int64
statusCode          Int64
dtype: object

## Дальше

Для EDA предстоит проверить пропуски и дубликаты, единицы времени, покрытие окна вокруг сигнала, согласованность сервисов между источниками и изменения после сбоя. Текущий ноутбук проверяет загрузку данных; выводов об их качестве пока нет.